In [243]:
import pandas as pd
import os
import glob
import yfinance as yf

In [253]:
df = pd.read_csv("new_datasets/nyse_yearly.csv")

df

,Year,sector,avg_return,volatility,avg_adj_close,avg_volume,avg_range_pct,sector_growth_yoy
0,1981,Aerospace & Defense,0.000432,0.023906,2.653802,6.184978e+05,-42.156635,16.318536
1,1982,Aerospace & Defense,0.001780,0.021678,2.847777,8.337740e+05,-51.924801,7.309339
2,1983,Aerospace & Defense,0.001562,0.017867,4.574161,7.581161e+05,-51.906654,60.622133
3,1984,Aerospace & Defense,0.000135,0.017482,4.573581,7.398553e+05,-53.946624,-0.012687
4,1985,Aerospace & Defense,0.000738,0.015115,5.085300,7.350666e+05,-57.488832,11.188594
...,...,...,...,...,...,...,...,...
1548,2018,Utilities,0.000052,0.015634,39.233629,1.463639e+06,-11.430617,2.808665
1549,2019,Utilities,0.001012,0.015687,46.301000,1.362915e+06,-8.762582,18.013555
1550,2020,Utilities,0.000304,0.031313,45.659141,1.588230e+06,-6.115123,-1.386275
1551,2021,Utilities,0.000670,0.015158,51.297075,1.311542e+06,-2.985893,12.347877


In [239]:
import pandas as pd

# Load your monthly datasets
dfs = [
    pd.read_csv("nyse_yearly.csv"),
    pd.read_csv("nasdaq_yearly.csv"),
    pd.read_csv("sp500_yearly.csv"),
    pd.read_csv("forbes2000_yearly.csv")
]

# Names for saving files
names = ['nyse_yearly', 'nasdaq_yearly', 'sp500_yearly', 'forbes2000_yearly']

# Convert 'YearMonth' to datetime
for df in dfs:
    df['Year'] = pd.to_datetime(df['Year'], format='%Y')

# Get all unique sectorsa
all_sectors = set().union(*[set(df['sector'].unique()) for df in dfs])

# Find common YearMonth per sector
common_yearmonths_per_sector = {}
for sector in all_sectors:
    yearmonths_list = [set(df[df['sector'] == sector]['Year']) for df in dfs]
    common_yearmonths = set.intersection(*yearmonths_list)
    common_yearmonths_per_sector[sector] = common_yearmonths

# Filter each dataframe, sort, and save
for df, name in zip(dfs, names):
    df_filtered = pd.DataFrame()
    for sector, yearmonths in common_yearmonths_per_sector.items():
        df_sector = df[(df['sector'] == sector) & (df['Year'].isin(yearmonths))]
        df_filtered = pd.concat([df_filtered, df_sector], ignore_index=True)
    
    # Sort by sector and YearMonth
    df_filtered.sort_values(by=['sector', 'Year'], inplace=True)
    
    # Convert YearMonth back to 'YYYY-MM' string format
    df_filtered['Year'] = df_filtered['Year'].dt.strftime('%Y')
    
    # Save each filtered dataset
    output_filename = f"{name}.csv"
    df_filtered.to_csv(output_filename, index=False)
    print(f"Saved: {output_filename}")


Saved: nyse_yearly.csv
Saved: nasdaq_yearly.csv
Saved: sp500_yearly.csv
Saved: forbes2000_yearly.csv


In [241]:
import pandas as pd

# Load your monthly datasets
dfs = [
    pd.read_csv("nyse_monthly.csv"),
    pd.read_csv("nasdaq_monthly.csv"),
    pd.read_csv("sp500_monthly.csv"),
    pd.read_csv("forbes2000_monthly.csv")
]

# Names for saving files
names = ['nyse_monthly', 'nasdaq_monthly', 'sp500_monthly', 'forbes2000_monthly']

# Convert 'YearMonth' to datetime (month precision)
for df in dfs:
    df['YearMonth'] = pd.to_datetime(df['YearMonth'], format='%Y-%m')

# Get all unique sectors
all_sectors = set().union(*[set(df['sector'].unique()) for df in dfs])

# Find common YearMonth per sector
common_months_per_sector = {}
for sector in all_sectors:
    months_list = [set(df[df['sector'] == sector]['YearMonth']) for df in dfs]
    common_months = set.intersection(*months_list)
    common_months_per_sector[sector] = common_months

# Filter each dataframe, sort, and save
for df, name in zip(dfs, names):
    df_filtered = pd.DataFrame()
    for sector, months in common_months_per_sector.items():
        df_sector = df[(df['sector'] == sector) & (df['YearMonth'].isin(months))]
        df_filtered = pd.concat([df_filtered, df_sector], ignore_index=True)
    
    # Sort by sector and YearMonth
    df_filtered.sort_values(by=['sector', 'YearMonth'], inplace=True)
    
    # Convert YearMonth back to 'YYYY-MM' string format
    df_filtered['YearMonth'] = df_filtered['YearMonth'].dt.strftime('%Y-%m')
    
    # Save each filtered dataset
    output_filename = f"{name}.csv"
    df_filtered.to_csv(output_filename, index=False)
    print(f"Saved: {output_filename}")

Saved: nyse_monthly.csv
Saved: nasdaq_monthly.csv
Saved: sp500_monthly.csv
Saved: forbes2000_monthly.csv


In [135]:
import os
import glob
import pandas as pd

# Relative path to the CSV folder
folder_path = "initial_datasets/companies/stock_market_data/forbes2000/csv"
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

print(f"Found {len(csv_files)} CSV files")

all_data = []

for file in csv_files:
    ticker = os.path.basename(file).split(".")[0]
    df = pd.read_csv(file)
    
    # Normalize column names
    df.columns = [col.strip().lower() for col in df.columns]
    
    if 'date' not in df.columns:
        print(f"'Date' column not found in {file}, skipping")
        continue
    
    df.rename(columns={'date': 'Date'}, inplace=True)
    df['company'] = ticker
    all_data.append(df)

if len(all_data) == 0:
    raise ValueError("No dataframes were added. Check your CSV files and column names.")

# Merge all CSVs into one DataFrame
big_df = pd.concat(all_data, ignore_index=True)
big_df['Date'] = pd.to_datetime(big_df['Date'])
big_df = big_df.sort_values(['Date', 'company']).reset_index(drop=True)

big_df

Found 1076 CSV files


C:\Users\Abel\AppData\Local\Temp\ipykernel_25964\3389795156.py:33: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  big_df['Date'] = pd.to_datetime(big_df['Date'])


,Date,low,open,volume,high,close,adjusted close,company
0,1970-01-02,3.447917,3.541667,276000.0,3.541667,3.458333,1.295931,CAT
1,1970-01-02,10.994125,11.099500,24555.0,11.204875,11.169750,0.416651,CNP
2,1970-01-02,0.683357,0.688495,1109377.0,0.689779,0.683357,0.454785,DIS
3,1970-01-02,18.297873,18.297873,7520.0,18.936171,18.936171,0.453444,DTE
4,1970-01-02,6.406250,6.406250,73200.0,6.750000,6.750000,0.253828,ED
...,...,...,...,...,...,...,...,...
5601966,2022-12-12,46.840000,47.470001,162578.0,47.720001,47.669998,47.669998,ZION
5601967,2022-12-12,33.850101,33.849998,13879.0,34.080002,33.850101,33.850101,ZNH
5601968,2022-12-12,51.500000,51.500000,648.0,51.500000,51.500000,51.500000,ZSHGY
5601969,2022-12-12,152.970001,154.070007,312714.0,154.470001,153.425003,153.425003,ZTS


In [137]:
import pandas as pd
import finnhub
import time

# --- Initialize Finnhub client ---
finnhub_client = finnhub.Client(api_key="d3e12qhr01qrd38selugd3e12qhr01qrd38selv0")  # replace with your key

# --- Get unique tickers ---
unique_tickers = big_df['company'].unique()
print(f"Fetching sectors for {len(unique_tickers)} unique tickers...")

# --- Fetch sectors ---
sector_dict = {}
for ticker in unique_tickers:
    try:
        profile = finnhub_client.company_profile2(symbol=ticker)
        sector = profile.get('finnhubIndustry', 'Unknown')
        sector_dict[ticker] = sector
    except Exception as e:
        print(f"Error fetching {ticker}: {e}")
        sector_dict[ticker] = 'Unknown'
    
    # Sleep a bit to avoid hitting API rate limits
    time.sleep(2)

big_df['sector'] = big_df['company'].map(sector_dict)

# --- Optional: check results ---
print(big_df[['company', 'sector']].drop_duplicates())

big_df

Fetching sectors for 1076 unique tickers...
        company              sector
0           CAT           Machinery
1           CNP           Utilities
2           DIS               Media
3           DTE           Utilities
4            ED           Utilities
...         ...                 ...
5129503     MON             Unknown
5209751     HTZ         Road & Rail
5293732      PX  Financial Services
5595268    INTL             Unknown
5601353     CBX             Unknown

[1076 rows x 2 columns]


,Date,low,open,volume,high,close,adjusted close,company,sector
0,1970-01-02,3.447917,3.541667,276000.0,3.541667,3.458333,1.295931,CAT,Machinery
1,1970-01-02,10.994125,11.099500,24555.0,11.204875,11.169750,0.416651,CNP,Utilities
2,1970-01-02,0.683357,0.688495,1109377.0,0.689779,0.683357,0.454785,DIS,Media
3,1970-01-02,18.297873,18.297873,7520.0,18.936171,18.936171,0.453444,DTE,Utilities
4,1970-01-02,6.406250,6.406250,73200.0,6.750000,6.750000,0.253828,ED,Utilities
...,...,...,...,...,...,...,...,...,...
5601966,2022-12-12,46.840000,47.470001,162578.0,47.720001,47.669998,47.669998,ZION,Banking
5601967,2022-12-12,33.850101,33.849998,13879.0,34.080002,33.850101,33.850101,ZNH,Unknown
5601968,2022-12-12,51.500000,51.500000,648.0,51.500000,51.500000,51.500000,ZSHGY,Retail
5601969,2022-12-12,152.970001,154.070007,312714.0,154.470001,153.425003,153.425003,ZTS,Pharmaceuticals


In [141]:
#big_df.to_csv("sector_forbes2000.csv", index=False)

In [46]:
# Load your CSV
df = pd.read_csv("initial_datasets/sector_sp500.csv")

df

,Date,low,open,volume,high,close,adjusted close,company,sector
0,1970-01-02,30.125000,0.000000,10300.0,31.000000,30.625000,0.925579,AEP,Utilities
1,1970-01-02,0.925926,0.925926,634838.0,0.979424,0.979424,0.290567,BA,Aerospace & Defense
2,1970-01-02,3.447917,3.541667,276000.0,3.541667,3.458333,1.295932,CAT,Machinery
3,1970-01-02,10.994125,11.099500,24555.0,11.204875,11.169750,0.416651,CNP,Utilities
4,1970-01-02,0.683357,0.688495,1109377.0,0.689779,0.683357,0.454785,DIS,Media
...,...,...,...,...,...,...,...,...,...
3265995,2022-12-12,110.959999,111.779999,136423.0,111.661003,111.231003,111.231003,XYL,Machinery
3265996,2022-12-12,126.980003,127.720001,345300.0,128.429993,128.320007,128.320007,YUM,"Hotels, Restaurants & Leisure"
3265997,2022-12-12,123.669998,124.370003,320440.0,125.480003,125.190002,125.190002,ZBH,Health Care
3265998,2022-12-12,46.840000,47.470001,160305.0,47.720001,47.709999,47.709999,ZION,Banking


In [44]:
# Load your CSV
df = pd.read_csv("new_datasets/sp500_yearly.csv")

df

,sector,Year,avg_return,volatility,mean_growth,avg_volume,sector_growth_yoy
0,Aerospace & Defense,1981,-0.054173,2.079697,-28.256661,9.909522e+05,364.548790
1,Aerospace & Defense,1982,0.161743,2.215726,45.377264,1.209105e+06,-260.589617
2,Aerospace & Defense,1983,0.153790,1.793147,55.440340,1.176966e+06,22.176471
3,Aerospace & Defense,1984,0.072850,1.758487,26.684957,1.275523e+06,-51.867257
4,Aerospace & Defense,1985,0.105417,1.520711,34.507301,1.291611e+06,29.313685
...,...,...,...,...,...,...,...
1614,Utilities,2018,0.029859,1.267343,0.714133,3.200334e+06,-93.737200
1615,Utilities,2019,0.097765,0.990281,10.532299,2.736118e+06,1374.837732
1616,Utilities,2020,0.023637,2.820700,-6.377255,3.135625e+06,-160.549501
1617,Utilities,2021,0.068424,1.295333,-1.227320,2.595388e+06,-80.754729


In [129]:
import numpy as np

numeric_cols = nasdaq_yearly.select_dtypes(include=[np.number])

inf_mask = np.isinf(numeric_cols)
has_infs = inf_mask.any().any() 

print(has_infs)

inf_columns = numeric_cols.columns[inf_mask.any()]
print(list(inf_columns))

False
[]


In [131]:
nasdaq_yearly.to_csv("nasdaq_yearly_2.csv", index=False)

In [211]:
import pandas as pd
import numpy as np

# Load CSV
big_df = pd.read_csv("initial_datasets/sector_sp500.csv")

# Ensure Date is datetime and extract Year
big_df['Date'] = pd.to_datetime(big_df['Date'], errors='coerce')
big_df = big_df.dropna(subset=['Date'])
big_df['Year'] = big_df['Date'].dt.year

# Compute company-level daily returns
big_df['daily_return'] = big_df.groupby('company')['adjusted close'].pct_change()

# Aggregate to sector level per year
sp_500_yearly = big_df.groupby(['sector', 'Year']).agg(
    avg_return=('daily_return', 'mean'),           # average return
    volatility=('daily_return', 'std'),            # standard deviation of returns
    avg_open=('open', 'mean'),
    avg_adj_close=('adjusted close', 'mean'),
    avg_volume=('volume', 'mean')                  # average volume
).reset_index()

# Sort and compute YoY growth
sp_500_yearly = sp_500_yearly.sort_values(['sector', 'Year'])
sp_500_yearly['sector_growth_yoy'] = sp_500_yearly.groupby('sector')['avg_adj_close'].pct_change() * 100
sp_500_yearly['avg_range_pct'] = ((sp_500_yearly['avg_adj_close'] - sp_500_yearly['avg_open']) / sp_500_yearly['avg_open']) * 100

sp_500_yearly = sp_500_yearly[['Year', 'sector', 'avg_return', 'volatility', 'avg_adj_close', 'avg_volume',
                               'avg_range_pct','sector_growth_yoy']]

# --- Final clean ---
sp_500_yearly.replace([np.inf, -np.inf], np.nan, inplace=True)
sp_500_yearly = sp_500_yearly.dropna().reset_index(drop=True)

sp_500_yearly

C:\Users\Abel\AppData\Local\Temp\ipykernel_25964\2926757723.py:13: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  big_df['daily_return'] = big_df.groupby('company')['adjusted close'].pct_change()


,Year,sector,avg_return,volatility,avg_adj_close,avg_volume,avg_range_pct,sector_growth_yoy
0,1971,Aerospace & Defense,0.001219,0.022792,0.789679,6.777087e+05,-76.095117,47.460831
1,1972,Aerospace & Defense,0.001052,0.020245,0.915482,5.991139e+05,-75.597485,15.931018
2,1973,Aerospace & Defense,-0.001032,0.026701,0.986771,4.324499e+05,-64.520449,7.787018
3,1974,Aerospace & Defense,-0.000428,0.027523,0.780837,3.763382e+05,-60.776171,-20.869531
4,1975,Aerospace & Defense,0.002285,0.024695,0.890872,6.582739e+05,-55.886933,14.091966
...,...,...,...,...,...,...,...,...
1995,2018,Utilities,0.000299,0.012673,51.417476,3.200334e+06,-13.298413,3.897532
1996,2019,Utilities,0.000978,0.009903,63.293588,2.736118e+06,-10.266131,23.097423
1997,2020,Utilities,0.000236,0.028207,65.853736,3.135625e+06,-7.406435,4.044877
1998,2021,Utilities,0.000684,0.012953,71.339260,2.595388e+06,-4.287309,8.329860


In [213]:
sp_500_yearly.to_csv("sp500_yearly.csv", index=False)

In [215]:
import pandas as pd
import numpy as np

# Load CSV
big_df = pd.read_csv("initial_datasets/sector_nyse.csv")

# Ensure Date is datetime and extract Year
big_df['Date'] = pd.to_datetime(big_df['Date'], errors='coerce')
big_df = big_df.dropna(subset=['Date'])
big_df['Year'] = big_df['Date'].dt.year

# Compute company-level daily returns
big_df['daily_return'] = big_df.groupby('company')['adjusted close'].pct_change()

# Aggregate to sector level per year
nyse_yearly = big_df.groupby(['sector', 'Year']).agg(
    avg_return=('daily_return', 'mean'),           # average return
    volatility=('daily_return', 'std'),            # standard deviation of returns
    avg_open=('open', 'mean'),
    avg_adj_close=('adjusted close', 'mean'),
    avg_volume=('volume', 'mean')                  # average volume
).reset_index()

# Sort and compute YoY growth
nyse_yearly = nyse_yearly.sort_values(['sector', 'Year'])
nyse_yearly['sector_growth_yoy'] = nyse_yearly.groupby('sector')['avg_adj_close'].pct_change() * 100
nyse_yearly['avg_range_pct'] = ((nyse_yearly['avg_adj_close'] - nyse_yearly['avg_open']) / nyse_yearly['avg_open']) * 100

nyse_yearly = nyse_yearly[['Year', 'sector', 'avg_return', 'volatility', 'avg_adj_close', 'avg_volume',
                               'avg_range_pct','sector_growth_yoy']]

# --- Final clean ---
nyse_yearly.replace([np.inf, -np.inf], np.nan, inplace=True)
nyse_yearly = nyse_yearly.dropna().reset_index(drop=True)

nyse_yearly

C:\Users\Abel\AppData\Local\Temp\ipykernel_25964\2858109076.py:13: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  big_df['daily_return'] = big_df.groupby('company')['adjusted close'].pct_change()


,Year,sector,avg_return,volatility,avg_adj_close,avg_volume,avg_range_pct,sector_growth_yoy
0,1971,Aerospace & Defense,0.001054,0.019299,1.089751,4.388016e+05,-76.532067,52.320809
1,1972,Aerospace & Defense,0.000819,0.018350,1.255673,3.607126e+05,-76.077476,15.225667
2,1973,Aerospace & Defense,-0.000360,0.023663,1.555612,2.768958e+05,-65.206823,23.886720
3,1974,Aerospace & Defense,-0.000752,0.024689,1.455534,2.262070e+05,-56.528906,-6.433391
4,1975,Aerospace & Defense,0.001081,0.024978,1.367046,4.382267e+05,-56.528894,-6.079400
...,...,...,...,...,...,...,...,...
2015,2018,Utilities,0.000052,0.015634,39.233629,1.463639e+06,-11.430617,2.808665
2016,2019,Utilities,0.001012,0.015687,46.301000,1.362915e+06,-8.762582,18.013555
2017,2020,Utilities,0.000304,0.031313,45.659141,1.588230e+06,-6.115123,-1.386275
2018,2021,Utilities,0.000670,0.015158,51.297075,1.311542e+06,-2.985893,12.347877


In [217]:
nyse_yearly.to_csv("nyse_yearly.csv", index=False)

In [219]:
import pandas as pd
import numpy as np

# Load CSV
big_df = pd.read_csv("initial_datasets/sector_nasdaq.csv")

# Ensure Date is datetime and extract Year
big_df['Date'] = pd.to_datetime(big_df['Date'], errors='coerce')
big_df = big_df.dropna(subset=['Date'])
big_df['Year'] = big_df['Date'].dt.year

# Compute company-level daily returns
big_df['daily_return'] = big_df.groupby('company')['adjusted close'].pct_change()

# Aggregate to sector level per year
nasdaq_yearly = big_df.groupby(['sector', 'Year']).agg(
    avg_return=('daily_return', 'mean'),           # average return
    volatility=('daily_return', 'std'),            # standard deviation of returns
    avg_open=('open', 'mean'),
    avg_adj_close=('adjusted close', 'mean'),
    avg_volume=('volume', 'mean')                  # average volume
).reset_index()

# Sort and compute YoY growth
nasdaq_yearly = nasdaq_yearly.sort_values(['sector', 'Year'])
nasdaq_yearly['sector_growth_yoy'] = nasdaq_yearly.groupby('sector')['avg_adj_close'].pct_change() * 100
nasdaq_yearly['avg_range_pct'] = ((nasdaq_yearly['avg_adj_close'] - nasdaq_yearly['avg_open']) / nasdaq_yearly['avg_open']) * 100

nasdaq_yearly = nasdaq_yearly[['Year', 'sector', 'avg_return', 'volatility', 'avg_adj_close', 'avg_volume',
                               'avg_range_pct','sector_growth_yoy']]

# --- Final clean ---
nasdaq_yearly.replace([np.inf, -np.inf], np.nan, inplace=True)
nasdaq_yearly = nasdaq_yearly.dropna().reset_index(drop=True)

nasdaq_yearly

C:\Users\Abel\AppData\Local\Temp\ipykernel_25964\2372163507.py:13: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  big_df['daily_return'] = big_df.groupby('company')['adjusted close'].pct_change()


,Year,sector,avg_return,volatility,avg_adj_close,avg_volume,avg_range_pct,sector_growth_yoy
0,1981,Aerospace & Defense,0.000489,0.029250,2.843158,33600.464427,7438.366256,93.810475
1,1982,Aerospace & Defense,-0.000061,0.028853,1.860698,23356.367589,3417.508427,-34.555248
2,1983,Aerospace & Defense,0.002308,0.029972,2.350066,38429.274704,4578.983891,26.300244
3,1984,Aerospace & Defense,-0.000392,0.023715,1.896223,20457.100791,3049.571476,-19.311908
4,1985,Aerospace & Defense,0.001664,0.025047,20.600959,24088.324547,9.760066,986.420639
...,...,...,...,...,...,...,...,...
1780,2018,Utilities,0.000229,0.019970,26.745065,51777.529880,-6.874375,4.598996
1781,2019,Utilities,0.000597,0.016570,30.901636,61039.325397,-4.778121,15.541450
1782,2020,Utilities,0.000138,0.036674,30.866095,99747.272727,-2.910744,-0.115013
1783,2021,Utilities,0.000791,0.025056,36.513964,130110.357143,-0.792462,18.297970


In [227]:
nasdaq_yearly.to_csv("nasdaq_yearly.csv", index=False)

In [229]:
import pandas as pd
import numpy as np

# Load CSV
big_df = pd.read_csv("initial_datasets/sector_forbes2000.csv")

# Ensure Date is datetime and extract Year
big_df['Date'] = pd.to_datetime(big_df['Date'], errors='coerce')
big_df = big_df.dropna(subset=['Date'])
big_df['Year'] = big_df['Date'].dt.year

# Compute company-level daily returns
big_df['daily_return'] = big_df.groupby('company')['adjusted close'].pct_change()

# Aggregate to sector level per year
forbes2000_yearly = big_df.groupby(['sector', 'Year']).agg(
    avg_return=('daily_return', 'mean'),           # average return
    volatility=('daily_return', 'std'),            # standard deviation of returns
    avg_close=('close', 'mean'),
    avg_open=('open', 'mean'),
    avg_adj_close=('adjusted close', 'mean'),
    avg_volume=('volume', 'mean')                  # average volume
).reset_index()

# Sort and compute YoY growth
forbes2000_yearly = forbes2000_yearly.sort_values(['sector', 'Year'])
forbes2000_yearly['sector_growth_yoy'] = forbes2000_yearly.groupby('sector')['avg_adj_close'].pct_change() * 100
forbes2000_yearly['avg_range_pct'] = ((forbes2000_yearly['avg_adj_close'] - forbes2000_yearly['avg_open']) / forbes2000_yearly['avg_open']) * 100

forbes2000_yearly = forbes2000_yearly[['Year', 'sector', 'avg_return', 'volatility', 'avg_adj_close', 'avg_volume',
                               'avg_range_pct','sector_growth_yoy']]

# --- Final clean ---
forbes2000_yearly.replace([np.inf, -np.inf], np.nan, inplace=True)
forbes2000_yearly = forbes2000_yearly.dropna().reset_index(drop=True)

forbes2000_yearly

C:\Users\Abel\AppData\Local\Temp\ipykernel_25964\2397346835.py:13: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  big_df['daily_return'] = big_df.groupby('company')['adjusted close'].pct_change()


,Year,sector,avg_return,volatility,avg_adj_close,avg_volume,avg_range_pct,sector_growth_yoy
0,1971,Aerospace & Defense,0.001054,0.019299,1.089171,4.388016e+05,-76.544556,52.409921
1,1972,Aerospace & Defense,0.000819,0.018350,1.255028,3.607126e+05,-76.089778,15.227736
2,1973,Aerospace & Defense,-0.000604,0.025363,1.261651,2.882120e+05,-64.285830,0.527717
3,1974,Aerospace & Defense,-0.001046,0.028463,0.982681,2.517687e+05,-60.322504,-22.111437
4,1975,Aerospace & Defense,0.002302,0.024184,1.095648,4.804783e+05,-54.807354,11.495770
...,...,...,...,...,...,...,...,...
1958,2018,Utilities,0.015652,0.493859,29.445924,1.145380e+06,-15.090902,7.237412
1959,2019,Utilities,0.014617,0.785444,34.600186,9.723706e+05,-11.600144,17.504159
1960,2020,Utilities,0.005391,0.278255,36.103021,1.099947e+06,-9.396866,4.343431
1961,2021,Utilities,0.010996,0.799038,39.887573,9.021051e+05,-7.420946,10.482648


In [231]:
forbes2000_yearly.to_csv("forbes2000_yearly.csv", index=False)

In [235]:
import pandas as pd
import numpy as np

def preprocess_sector_monthly(csv_path):
    """
    Preprocess a sector stock CSV into monthly aggregated features:
    avg_return, avg_adj_close, avg_volume, avg_range, sector_growth_yoy.
    Returns a cleaned DataFrame.
    """
    # --- Load CSV ---
    df = pd.read_csv(csv_path)
    
    # --- Ensure Date is datetime and extract year-month ---
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df = df.dropna(subset=['Date'])
    df['YearMonth'] = df['Date'].dt.to_period('M').astype(str)  # e.g., '2022-03'
    
    # --- Compute company-level daily returns ---
    df['daily_return'] = df.groupby('company')['adjusted close'].pct_change()
    
    # --- Aggregate features per sector-month ---
    monthly = df.groupby(['sector', 'YearMonth']).agg(
        avg_return=('daily_return', 'mean'),
        avg_close=('close', 'mean'),
        volatility=('daily_return', 'std'),            # standard deviation of returns
        avg_open=('open', 'mean'),
        avg_adj_close=('adjusted close', 'mean'),
        avg_volume=('volume', 'mean')
    ).reset_index()
    
    # --- Sort by sector and YearMonth ---
    monthly = monthly.sort_values(['sector', 'YearMonth']).reset_index(drop=True)
    
    # --- Compute sector_growth_monthly ---
    monthly['sector_growth_yoy'] = monthly.groupby('sector')['avg_adj_close'].pct_change() * 100
    
    # --- Add avg_range ---
    monthly['avg_range_pct'] = ((monthly['avg_adj_close'] - monthly['avg_open']) / monthly['avg_open']) * 100

    monthly = monthly[['YearMonth', 'sector', 'avg_return', 'volatility', 'avg_adj_close', 'avg_volume',
                               'avg_range_pct','sector_growth_yoy']]
    
    # --- Clean infinite and NaN values ---
    monthly.replace([np.inf, -np.inf], np.nan, inplace=True)
    monthly = monthly.dropna().reset_index(drop=True)
    
    return monthly

nasdaq_monthly = preprocess_sector_monthly("initial_datasets/sector_nasdaq.csv")
nyse_monthly   = preprocess_sector_monthly("initial_datasets/sector_nyse.csv")
sp500_monthly  = preprocess_sector_monthly("initial_datasets/sector_sp500.csv")
forbes_monthly = preprocess_sector_monthly("initial_datasets/sector_forbes2000.csv")

C:\Users\Abel\AppData\Local\Temp\ipykernel_25964\4218708183.py:19: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df['daily_return'] = df.groupby('company')['adjusted close'].pct_change()
C:\Users\Abel\AppData\Local\Temp\ipykernel_25964\4218708183.py:19: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df['daily_return'] = df.groupby('company')['adjusted close'].pct_change()
C:\Users\Abel\AppData\Local\Temp\ipykernel_25964\4218708183.py:19: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill i

In [237]:
#nasdaq_monthly.to_csv("nasdaq_monthly.csv", index=False)
#nyse_monthly.to_csv("nyse_monthly.csv", index=False)
#sp500_monthly.to_csv("sp500_monthly.csv", index=False)
#forbes_monthly.to_csv("forbes2000_monthly.csv", index=False)

In [ ]:
import pandas as pd
import finnhub
import time

finnhub_client = finnhub.Client(api_key="can_get_custom_key_after_signing_up_to_Finnhub")

sp500_df = pd.read_csv("sp500_monthly.csv")
sp500_df['Date'] = pd.to_datetime(sp500_df['Date'], errors='coerce')
sp500_df = sp500_df.dropna(subset=['Date'])
sp500_df['Year'] = sp500_df['Date'].dt.year

tickers = sp500_df['company'].unique()
print(f"Fetching sectors for {len(tickers)} tickers...")

sector_dict = {}

for ticker in tickers:
    try:
        profile = finnhub_client.company_profile2(symbol=ticker)
        sector = profile.get('finnhubIndustry', 'Unknown')
        sector_dict[ticker] = sector
    except Exception as e:
        print(f"Error fetching {ticker}: {e}")
        sector_dict[ticker] = 'Unknown'
    
    time.sleep(2)

sp500_df['sector'] = sp500_df['company'].map(sector_dict)

sector_lookup = sp500_df[['company', 'sector']].drop_duplicates().reset_index(drop=True)
print(sector_lookup.head())